In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import re

#データの各項目の説明txtを開く
with open('/kaggle/input/house-prices-advanced-regression-techniques/data_description.txt', 'r') as f:
    lines = f.readlines()
desc = []
for line in lines:
    match = re.match(r"^([A-Za-z0-9_]+):\s*(.*)", line)
    if match:
        desc.append(match.groups())

data_description = pd.DataFrame(desc, columns=['Feature', 'Description'])

#CSV形式で出力
data_description.to_csv('data_description_full.csv', index=False, encoding='utf-8')

# pandas の表示設定を変更
pd.set_option('display.max_rows', None)  # 行をすべて表示
pd.set_option('display.max_columns', None)  # 列をすべて表示
pd.set_option('display.width', None)  # 横幅制限を解除
pd.set_option('display.max_colwidth', None)  # 列の文字数制限を解除


#改めてpandasで読み込み
pd.read_csv('data_description_full.csv')

In [ ]:
#学習データとテストデータをインポート
train = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

In [ ]:
#テストデータの個数を確認
test.count()

In [ ]:
#学習データの状況を確認
#Idもインデックス化

train = train.set_index('Id')

train.head(5)

In [ ]:
#学習データの統計値を確認
train.describe()

In [ ]:
#学習データの欠損値を確認
train.isnull().sum()

In [ ]:
#PoolQC, Fence, MiscFeature, FireplcaQu, Alleyで「NAN」となっている項目は、そもそもたとえばPoolなどが存在しないということであるため、Noneに置き換える
train['PoolQC'] = train['PoolQC'].fillna('None')
train['Fence'] = train['Fence'].fillna('None')
train['MiscFeature'] = train['MiscFeature'].fillna('None')
train['FireplaceQu'] = train['FireplaceQu'].fillna('None')
train['Alley'] = train['Alley'].fillna('None')

In [ ]:
#テストデータも同様に
test['PoolQC'] = test['PoolQC'].fillna('None')
test['Fence'] = test['Fence'].fillna('None')
test['MiscFeature'] = test['MiscFeature'].fillna('None')
test['FireplaceQu'] = test['FireplaceQu'].fillna('None')
test['Alley'] = test['Alley'].fillna('None')

In [ ]:
#現在の学習データの欠損値の個数を確認
#ここで上記項目の欠損値が「0」となっていれば置き換え成功
train.isnull().sum()

In [ ]:
#相関図を作成する
import matplotlib.pyplot as plt
import seaborn as sns

# 数値列のみで相関行列を作成
corr = train.corr(numeric_only=True)

plt.figure(figsize=(40,40))
sns.heatmap(corr,annot=True)

In [ ]:
#次にLotFrontageから見てみる
#LotFrontageは「Linear feet of street connected to property」
#欠損値となっているところのデータを抽出

LF_NA = train[train['LotFrontage'].isnull()]
LF_NA.head()

In [ ]:
#データの中身を確認

train['LotFrontage'].unique()

In [ ]:
#LotFrontageがNAとなっているときの特徴がないかを、統計量から確認
LF_NA.describe()

In [ ]:
#LotFrontage が NA かどうかでグループ化
#そのうえで、LotFrontageがNAのときとそうでないときで、LotFrontage以外の統計量に差が出るかを確認
#もし目立った差分がなければ、単純なデータの欠損と考えられる
#この場合はLotFrontageと相関のある項目で欠損値を埋める
#False：NA以外、True：NA


train.groupby(train['LotFrontage'].isna()).describe()

In [ ]:
#改めてLotfrontageと相関の高い項目を確認
target_col = 'LotFrontage' 

corr_series = train.corr(numeric_only=True)[target_col].sort_values(ascending=False)
print(corr_series)

In [ ]:
#Lotfrontageと相関を持ち、かつ項目として関連性もある「lotArea」との推移を確認
#上記2項目を散布図で見てみる

x = 'LotFrontage'
y = 'LotArea'

plt.figure(figsize=(10,10))
sns.regplot(x=train[x], y=train[y], ci=None, line_kws={'color':'red'})
plt.title(f"{x} vs {y}")
plt.xlabel(x)
plt.ylabel(y)
plt.grid(True)
plt.show()

In [ ]:
#数値的に回帰がどの程度かを確認する
from sklearn.linear_model import LinearRegression

x = 'LotFrontage'
y = 'LotArea'

#回帰分析用にいったん欠損値を削除したデータを作成
df = train[[x, y]].dropna()

#回帰モデルを作成・学習
model = LinearRegression()
model.fit(df[[x]], df[y])

#結果を表示
slope = model.coef_[0]     # 傾き
intercept = model.intercept_  # 切片

print(f"傾き（Xが1増えるとYがどのくらい変わるか）: {slope:.3f}")
print(f"切片（X=0のときのY）: {intercept:.3f}")

In [ ]:
#上記の結果をもとにLotFrontageの欠損値を埋める
target = 'LotFrontage'
features = ['LotArea']

#欠損なしの行で学習データを作成
df_train = train[train[target].notnull()]
df_pred = train[train[target].isnull()]

#モデル学習
model = LinearRegression()
model.fit(df_train[features], df_train[target])

#予測値を埋め込み
train.loc[df_pred.index, target] = model.predict(df_pred[features])

In [ ]:
#テストデータにも同様に実施
target = 'LotFrontage'
features = ['LotArea']

#欠損なしの行で学習データを作成
df_test = test[test[target].notnull()]
df_pred = test[test[target].isnull()]

#モデル学習
model = LinearRegression()
model.fit(df_test[features], df_test[target])

#予測値を埋め込み
test.loc[df_pred.index, target] = model.predict(df_pred[features])

In [ ]:
#LotFrontageの欠損値が埋まっているかを確認
train.isnull().sum()

In [ ]:
#次にMasVnrTypeを埋める
#MasVnrTypeの欠損値が出ているところをまずは確認
MV_NA = train[train['MasVnrType'].isnull()]
MV_NA.head(20)

In [ ]:
#MasVnrTypeがnanとなるところは、MasVnrAreaが「0」つまりMasonry veneerが使われていないからという可能性がある
#MasVnrType：Masonry veneer type
#MasVnrTypeがnanの時とそうでないときで統計量の差を確認（Masonry Veneerにフォーカスする）

train.groupby(train['MasVnrType'].isna()).describe()

In [ ]:
#統計量でMasVnrTypeがnanのときに、MasVnrAreaが「0」以外のケースもあった
#このケースが発生しているデータを確認
train[(train['MasVnrType'].isna()) & (train['MasVnrArea'] > 0)]

In [ ]:
#上記５つのデータを除いたMasVnrTypeについては、Masonry Veneerがそもそも使われていないため、Noneと置き換える
condition = (train['MasVnrArea'] == 0)
train.loc[condition, 'MasVnrType'] = train.loc[condition, 'MasVnrType'].fillna('None')

In [ ]:
#テストも同様に実施
condition = (test['MasVnrArea'] == 0)
test.loc[condition, 'MasVnrType'] = test.loc[condition, 'MasVnrType'].fillna('None')

In [ ]:
#現在の学習データの中身を確認する
train.isnull().sum()

In [ ]:
#MasVnrTypeのうち、いまだNANとなっている箇所を確認
M_N =train[train['MasVnrType'].isnull()]
M_N

In [ ]:
#MasVnrTypeのうち、MasVnrAreaがnanとなっている箇所は「そもそもMasVnrAreaがない」と考え、MasVnrAreaを0で埋め、MasVnrtypeを「None」とする
train['MasVnrArea'] = train['MasVnrArea'].fillna(0)
condition = (train['MasVnrArea'] == 0)
train.loc[condition, 'MasVnrType'] = train.loc[condition, 'MasVnrType'].fillna('None')

In [ ]:
#テストも同様に処理（念のため）
test['MasVnrArea'] = test['MasVnrArea'].fillna(0)
condition = (test['MasVnrArea'] == 0)
test.loc[condition, 'MasVnrType'] = test.loc[condition, 'MasVnrType'].fillna('None')

In [ ]:
#MasVnrTypeのうち、いまだNANとなっている箇所を確認
M_N2 =train[train['MasVnrType'].isnull()]
M_N2

In [ ]:
#MasVnrTypeのうち、MasVnrAreaが0よりも上の数値が出ている箇所は、MasVnrTypeの各データの統計量をみて、最頻値と出ているType（Noneを除く）を当てはめる
train.groupby('MasVnrType').count()

In [ ]:
#MasVnrTypeはBrkFaceとなっているため、念のためこの部分に絞って統計量を確認
train.groupby(train['MasVnrType'])['MasVnrArea'].describe()

In [ ]:
#MasVnrTypeのうち、MasVnrAreaが0よりも上の数値が出ている箇所をBrkFaceで埋める
train['MasVnrType'] = train['MasVnrType'].fillna('BrkFace')

In [ ]:
#テストも同様に実施
test['MasVnrType'] = test['MasVnrType'].fillna('BrkFace')

In [ ]:
#現状の学習データの欠損値を確認
train.isnull().sum()

In [ ]:
#次にelectrialの欠損値を埋めるか否かの判断を実施する
#まずはデータの中身を確認

train['Electrical'].unique()

In [ ]:
#ElectricalにてNANとなっているデータを確認
train[train['Electrical'].isnull()]

In [ ]:
#念のためNan以外のElectricalのデータの統計量を確認
train.groupby('Electrical').count()

In [ ]:
#今回はElectricalに直接関連する項目が見当たらないため、最頻値で埋める
train['Electrical'] = train['Electrical'].fillna('SBrkr')

In [ ]:
#現在の学習データを確認

train.isnull().sum()

In [ ]:
#Garage関連の欠損値を埋める方法を検討する
#まずは欠損値が発生しているところのデータを確認

G_NA = train[train['GarageType'].isnull()].head()
print(G_NA)

In [ ]:
#上記結果を見ると、Garage関係で欠損値となっているところはそもそも当てはまる項目がない（garage自体がない）ためにNAとなっている可能性がある
#念のためGarage関連の項目に欠損値が発生していないところで、統計量を見てみる（定量データの「GarageCars」で確認）
#ここで0となっている項目が欠損値の数と一致する場合、NA値となっているのはGarage自体がないからといえる

GC_0 = train[train['GarageCars']==0].count()
print(GC_0)

In [ ]:
#もう一度Garage関連の欠損値を確認する
train.isnull().sum()

In [ ]:
#学習データの中身も確認
train.head()

In [ ]:
#Garage関連の項目はGarageareaとGarageCarsが0（＝ガレージが存在しない）場合、NA値となる
#Garageが存在する場合は1、そうでない場合は0として、まずは選別

train['Garage_exists'] = (train['GarageYrBlt'].notna()).astype(int)
test['Garage_exists']  = (test['GarageYrBlt'].notna()).astype(int)

In [ ]:
#そのうえで最新の状態からの経過年数でGarageのの質を問う項目を作る
#まずは建築年のMax値を確認

G_M = train['GarageYrBlt'].max()
G_M

In [ ]:
#テストデータも同様に確認

G_M_T = test['GarageYrBlt'].max()
G_M_T

In [ ]:
#テストデータにおいて外れ値が存在していることを確認
#2207年はあり得ない値であるため修正する
#学習データの2010を基準とし、それ以降については外れ値とみなす

outliers = test[test['GarageYrBlt'] > 2010]
outliers

In [ ]:
#上記項目については、家の建築と同じタイミングで建築されたと考え、建築年を置き換え
test.loc[test['GarageYrBlt'] > 2010, 'GarageYrBlt'] = test['YearBuilt']

In [ ]:
#Garageの経過年数を埋める新規項目を作成
#ガレージがない場合と区別すべく2011を基準年とする
train['GarageAge'] = np.where(train['Garage_exists'] == 1,
                              2011 - train['GarageYrBlt'],
                              0)

test['GarageAge'] = np.where(test['Garage_exists'] == 1,
                             2011 - test['GarageYrBlt'],
                             0)

In [ ]:
#念のためデータを確認
train['GarageAge'].describe()

In [ ]:
#念のためデータを確認
test['GarageAge'].describe()

In [ ]:
#新規項目の作成が完了したため、GarageYrBltおよびGarage_existsは削除する
del train['GarageYrBlt']
del test['GarageYrBlt']
del train['Garage_exists']
del test['Garage_exists']

In [ ]:
#Garage関連の項目はGarageareaとGarageCarsが0（＝ガレージが存在しない）場合、NA値となる
#定性データのNA値を「NaN」で埋める

cols = ['GarageType','GarageFinish','GarageQual', 'GarageCond']

# 条件：ガレージが存在しない行
condition = (train['GarageArea'] == 0) & (train['GarageCars'] == 0)

# 該当する行のcols列をnp.nanで埋める
train.loc[condition, cols] = np.nan

#そのうえでNoneで埋める
train[cols] = train[cols].fillna('None')

In [ ]:
#欠損値を埋めることができているか確認
train.isnull().sum()

In [ ]:
#最後にBsmt関連の項目の欠損値を埋める
#まずは欠損値の発生しているデータを確認

B_N=train[train['BsmtQual'].isnull()]
B_N

In [ ]:
#上記結果から「BsmtFinSF1」（Type 1 finished square feet）が0、つまりbasementareaがないときに、bsmt関連の項目が欠損値となって表れている
#よって先ほどのGarageと同様に、定性データはNone、定量データは「0」で埋める
#なお「BsmtFinType2」（Rating of basement finished area (if multiple types））については、ほかの欠損値が37に対して、欠損値が38となっているため、Bsmtfintype1で欠損値がなく、Bsmtfintype2で欠損値が発生していると考える
#このためBsmntFintype1とBsmntFintype2は別々の条件で埋める
#まずはBsmntFintype1関連の欠損値を埋める
B1_cols = ['BsmtFinType1']

# 条件：BsmtFinSF1が0の行
condition = (train['BsmtFinSF1'] == 0)

# 該当する行のcols列をnp.nanで埋める
train.loc[condition, B1_cols] = np.nan

#そのうえでNoneで埋める
train[B1_cols] = train[B1_cols].fillna('None')


In [ ]:
#次にBsmntFintype2関連の欠損値を埋める
B2_cols = ['BsmtFinType2']

# 条件：BsmtFinSF2が0の行
condition = (train['BsmtFinSF2'] == 0)

# 該当する行のcols列をnp.nanで埋める
train.loc[condition, B2_cols] = np.nan

#そのうえでNoneで埋める
train[B2_cols] = train[B2_cols].fillna('None')

In [ ]:
#BsmtQual(Evaluates the height of the basement)とBsmtCond(Evaluates the general condition of the basement)は、どちらもBasementが存在しないために欠損値となっていると案が得る
#そのためBsmtFinSF1とBsmtFinSF2が「0」の場合という条件付きで欠損値を埋める

Bs_cols = ['BsmtQual','BsmtCond']

# 条件：Basementが存在しない行
condition = (train['BsmtFinSF1'] == 0) & (train['BsmtFinSF2'] == 0)

# 該当する行のcols列をnp.nanで埋める
train.loc[condition, Bs_cols] = np.nan

#そのうえでNoneで埋める
train[Bs_cols] = train[Bs_cols].fillna('None')

In [ ]:
#最後にBsmtExposureについて、まずどんな項目がデータに含まれているかを確認する
train['BsmtExposure'].unique()

In [ ]:
#BsmtExposureが「NA」となっているケースがどんなときかを調べるべく、データを抽出
BE_N =train[train['BsmtExposure'].isnull()]
BE_N

In [ ]:
#こちらもBasementがそもそも存在しないときにNAとなっている
#そのためBsmtFinSF1とBsmtFinSF2が「0」の場合という条件付きで欠損値を埋める

BE_cols = ['BsmtExposure']

# 条件：Basementが存在しない行
condition = (train['BsmtFinSF1'] == 0) & (train['BsmtFinSF2'] == 0)

# 該当する行のcols列をnp.nanで埋める
train.loc[condition, BE_cols] = np.nan

#そのうえでNoneで埋める
train[BE_cols] = train[BE_cols].fillna('None')

In [ ]:
#テストも同様に実施
BE_cols = ['BsmtExposure']

# 条件：Basementが存在しない行
condition = (test['BsmtFinSF1'] == 0) & (test['BsmtFinSF2'] == 0)

# 該当する行のcols列をnp.nanで埋める
test.loc[condition, BE_cols] = np.nan

#そのうえでNoneで埋める
test[BE_cols] = test[BE_cols].fillna('None')

In [ ]:
#現在の欠損値の有無を確認
train.isnull().sum()

In [ ]:
#欠損値処理が完了したため、統計量も把握する（定量データに限る）
#一部欠損値の穴埋めの関係でstdやmaxが異常に高いケースもあるが、その点はいったん無視
train.describe()

In [ ]:
#定量データには外れ値が存在する可能性も十分あるため、まずは外れ値の有無を確認する
#なお定性データに外れ値は存在しないと仮定し、数値列のみを抽出
#外れ値にも予測がつくもの（例：敷地面積が対象の家庭だけ広いため、大きくなっている）ものとそうでないものがあるため、見極めが大事
#予測がつく、かつ意義があると考えられる項目はそのまま残す
#そうでないものは処理方法を検討する
#この区別を上の統計量との兼ね合いで考える

num_cols = train.select_dtypes(include=['int64', 'float64']).columns

outlier_info = []

for col in num_cols:
    Q1 = train[col].quantile(0.25)
    Q3 = train[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((train[col] < lower) | (train[col] > upper)).sum()
    
    outlier_info.append({
        'Column': col,
        'Outlier_count': outliers,
        'Outlier_ratio(%)': round(outliers / len(train) * 100, 2)
    })

outlier_df = pd.DataFrame(outlier_info).sort_values('Outlier_count', ascending=False)
display(outlier_df)

In [ ]:
#基本的に今回の外れ値は有意性がある（もともと部屋の敷地が広いなどの理由が考えられる）ため、外れ値に対する処理は実施しない
#このまま分析を実施する
#現時点での相関を確認（数値データに限る）
plt.figure(figsize=(40,40))
sns.heatmap(train.corr(numeric_only=True), annot=True)

In [ ]:
#目的変数と相関が一番ある「OverallQual」のデータの中身を確認
train['OverallQual'].unique()

In [ ]:
#OverallQualityは「1-10」の評価でデータが作成されており、これがsalespriceにも反映されている可能性あり
#それぞれの評価別のsalepriceレンジを確認

train['SalePrice'].groupby(train['OverallQual']).describe()

In [ ]:
#全般的にOverallQualとSalePriceは比例的関係にあるが、最低ラインで若干の変動あり（ex：Quality2で最低価格が35311となっている一方で、Quality1で最低価格が39300となっている）
#この最低ラインに影響している項目をとらえるべく、OverallQualとその他の相関、および最低価格ライン上にあるデータと合わせて確認
#overallQualが1・2、9と10で逆転現象が起きている
#該当するデータを抽出して確認

O1 = train[(train['OverallQual']==1) & (train['SalePrice']==39300)]
O2 = train[(train['OverallQual']==2) & (train['SalePrice']==35311)]
O9 = train[(train['OverallQual']==9) & (train['SalePrice']==239000)]
O10 = train[(train['OverallQual']==10) & (train['SalePrice']==160000)]

In [ ]:
#そのうえでoverallQualが1・2、9と10で逆転現象が起きている理由を探る
pd.concat([O1.describe().T['mean'], O2.describe().T['mean']], axis=1, keys=['Qual1', 'Qual2'])

In [ ]:
#そのうえでoverallQualが9と10で逆転現象が起きている理由を探る
pd.concat([O9.describe().T['mean'], O10.describe().T['mean']], axis=1, keys=['Qual9', 'Qual10'])

In [ ]:
#salespriceとiverallqualityで逆転現象が起きるのは、Yearssoldの可能性もある？
#ただYearssoldとSalePriceの相関は低い（-0.02）
#念のため定性データとSalepriceおよびOverallQualityとの関係を見てみる

def correlation_ratio(categories, values):
    """カテゴリ（定性）と数値（定量）の関係を測る相関比ηを返す"""
    fcat, _ = pd.factorize(categories)
    mean_per_cat = np.array([values[fcat == i].mean() for i in range(len(np.unique(fcat)))])
    n_per_cat = np.array([np.sum(fcat == i) for i in range(len(np.unique(fcat)))])
    overall_mean = values.mean()
    numerator = np.sum(n_per_cat * (mean_per_cat - overall_mean)**2)
    denominator = np.sum((values - overall_mean)**2)
    return np.sqrt(numerator / denominator) if denominator != 0 else 0

# 定性データの列を抽出
cat_cols = train.select_dtypes(include='object').columns

# 目的変数（定量データ）
target = 'OverallQual'

# 各カテゴリ列との相関比を計算
results = []
for col in cat_cols:
    eta = correlation_ratio(train[col], train[target])
    results.append((col, eta))

# 結果をDataFrame化
corr_eta_df = pd.DataFrame(results, columns=['Column', 'Eta']).sort_values(by='Eta', ascending=False)
corr_eta_df.head()

In [ ]:
#salespriceとiverallqualityで逆転現象が起きるのは、Yearssoldの可能性もある？
#ただYearssoldとSalePriceの相関は低い（-0.02）
#念のため定性データとSalepriceおよびOverallQualityとの関係を見てみる

def correlation_ratio(categories, values):
    """カテゴリ（定性）と数値（定量）の関係を測る相関比ηを返す"""
    fcat, _ = pd.factorize(categories)
    mean_per_cat = np.array([values[fcat == i].mean() for i in range(len(np.unique(fcat)))])
    n_per_cat = np.array([np.sum(fcat == i) for i in range(len(np.unique(fcat)))])
    overall_mean = values.mean()
    numerator = np.sum(n_per_cat * (mean_per_cat - overall_mean)**2)
    denominator = np.sum((values - overall_mean)**2)
    return np.sqrt(numerator / denominator) if denominator != 0 else 0

# 定性データの列を抽出
cat_cols = train.select_dtypes(include='object').columns

# 目的変数（定量データ）
target = 'SalePrice'

# 各カテゴリ列との相関比を計算
results = []
for col in cat_cols:
    eta = correlation_ratio(train[col], train[target])
    results.append((col, eta))

# 結果をDataFrame化
corr_eta_df = pd.DataFrame(results, columns=['Column', 'Eta']).sort_values(by='Eta', ascending=False)
corr_eta_df.head()

In [ ]:
#SalesPice・OverallQualityにはNeighborhood・ExternalQualが関係している可能性がたかい
#この仮説の元、再度overallQualが1・2、9と10で逆転現象が起きている理由を探る
pd.concat([O1, O2])

In [ ]:
#SalesPice・OverallQualityにはNeighborhood・ExternalQualが関係している可能性がたかい
#この仮説の元、再度overallQualが1・2、9と10で逆転現象が起きている理由を探る
pd.concat([O9, O10])

In [ ]:
#ExternalConditionで逆転現象は見受けることができていない
#そのためNeighborhoodに絞ってみる
#Neighborhoodごとoverallquality・SalePriceの関係を見る
train.groupby('Neighborhood')[['OverallQual','SalePrice']].describe()

In [ ]:
#上記結果を見るとNeighborhoodのtypeとSalePriceに一定程度影響があり、一方でOverallQualityはNeighborhoodのtypeうんぬんに関連がないように見受けられる
#そのため今回OverallQualityの結果とSalePriceが逆転した理由は、Neighborhoodに関係があると発見
#もう一度定量データ側の相関を見る

plt.figure(figsize=(40,40))
sns.heatmap(train.corr(numeric_only=True), annot=True)

In [ ]:
#定性データとSalePriceの相関を確認する

def correlation_ratio(categories, values):
    """カテゴリ（定性）と数値（定量）の関係を測る相関比ηを返す"""
    fcat, _ = pd.factorize(categories)
    mean_per_cat = np.array([values[fcat == i].mean() for i in range(len(np.unique(fcat)))])
    n_per_cat = np.array([np.sum(fcat == i) for i in range(len(np.unique(fcat)))])
    overall_mean = values.mean()
    numerator = np.sum(n_per_cat * (mean_per_cat - overall_mean)**2)
    denominator = np.sum((values - overall_mean)**2)
    return np.sqrt(numerator / denominator) if denominator != 0 else 0

# 定性データの列を抽出
cat_cols = train.select_dtypes(include='object').columns

# 目的変数（定量データ）
target = 'SalePrice'

# 各カテゴリ列との相関比を計算
results = []
for col in cat_cols:
    eta = correlation_ratio(train[col], train[target])
    results.append((col, eta))

# 結果をDataFrame化
corr_eta_df = pd.DataFrame(results, columns=['Column', 'Eta']).sort_values(by='Eta', ascending=False)
corr_eta_df

In [ ]:
#今回のデータでは項目が多すぎるため、相関の低い項目は削除、もしくは説明変数同士の相関が強い場合（0.5以上）の場合は、目的変数との相関が強い方を残す
#まず定性データのうち、SaePriceとの相関が0.1未満のもの、もしくは他の定性・定量データで補完することで不要となる項目を削除する
#utilities：相関が極めて低く、かつ他の項目との補完の必要もないことから削除
#Street：Alleyと同じ内容（type of road access to property)であること、かつAlleyのSalepriceとの相関の方が高いことから、Alleyで補完できる
#Landslope：相関が極めて低い項目であることから、今回の機械学習では削除
#MiscFeature：相関が極めて低いことから、SalePriceを決める際の重要な項目ではないと判断し、削除
#BsmtFinType2：このBasementはすべての住宅に当てはまるわけではない項目（サブ的存在）であり、相関も低いことから削除
#Condition2：BsmtFintype2と同様に、サブ的存在の項目であり、相関も低いことから削除する

del train['Utilities']
del train['Street']
del train['LandSlope']
del train['MiscFeature']
del train['BsmtFinType2']
del train['Condition2']

In [ ]:
#テストデータも同様に実施
del test['Utilities']
del test['Street']
del test['LandSlope']
del test['MiscFeature']
del test['BsmtFinType2']
del test['Condition2']

In [ ]:
#次に定量データも削除する
#定量データについても定性データと同様に「SaePriceとの相関が0.1未満のもの、もしくは他の定性・定量データで補完することで不要となる項目」を削除する
#MSSubClass：相関が極めて低いことから、削除
#OverallCondition：相関が極めて低く、かつ似た項目である「overallQual」と逆の相関を示しており、機械学習に悪影響を及ぼすと認識されたため、削除
#BsmtFinSF2：相関が極めて低いことから、SalePriceを決める際の重要な指標となっていないと考えて削除
#BsmtHalfBath：相関が極めて低いことから、削除
#LowQualFinSF：相関が極めて低く、またLowQualであることの影響は「OverallQual」からSalePriceに反映されると考え、削除
#3SsnPorch：相関が低いため、SalePriceとも関連がないと考え削除
#PoolArea：Salepriceとの相関が低く、Poolの面積の広さでSalePriceが変わるわけではないと考え削除
#MiscVal：相関が極めて低く、SalePriceに影響を及ぼしているわけではないと考え削除
#Mosold, Yrsold：どちらも「Saleした年月」という結果にすぎず、Salepriceの考慮に入らないと考え削除

del train['MSSubClass']
del train['OverallCond']
del train['BsmtFinSF2']
del train['BsmtHalfBath']
del train['LowQualFinSF']
del train['3SsnPorch']
del train['PoolArea']
del train['MiscVal']
del train['MoSold']
del train['YrSold']

In [ ]:
#テストデータも同様に実施
del test['MSSubClass']
del test['OverallCond']
del test['BsmtFinSF2']
del test['BsmtHalfBath']
del test['LowQualFinSF']
del test['3SsnPorch']
del test['PoolArea']
del test['MiscVal']
del test['MoSold']
del test['YrSold']

In [ ]:
#現在のtrainの学習データを確認
train.head()

In [ ]:
#現在の定量データの相関を確認
plt.figure(figsize=(40,40))
sns.heatmap(train.corr(numeric_only=True), annot=True)

In [ ]:
#現状の定性データとSalePriceの相関を確認する

def correlation_ratio(categories, values):
    """カテゴリ（定性）と数値（定量）の関係を測る相関比ηを返す"""
    fcat, _ = pd.factorize(categories)
    mean_per_cat = np.array([values[fcat == i].mean() for i in range(len(np.unique(fcat)))])
    n_per_cat = np.array([np.sum(fcat == i) for i in range(len(np.unique(fcat)))])
    overall_mean = values.mean()
    numerator = np.sum(n_per_cat * (mean_per_cat - overall_mean)**2)
    denominator = np.sum((values - overall_mean)**2)
    return np.sqrt(numerator / denominator) if denominator != 0 else 0

# 定性データの列を抽出
cat_cols = train.select_dtypes(include='object').columns

# 目的変数（定量データ）
target = 'SalePrice'

# 各カテゴリ列との相関比を計算
results = []
for col in cat_cols:
    eta = correlation_ratio(train[col], train[target])
    results.append((col, eta))

# 結果をDataFrame化
corr_eta_df = pd.DataFrame(results, columns=['Column', 'Eta']).sort_values(by='Eta', ascending=False)
corr_eta_df

In [ ]:
#Alleyの中身が気になるため確認
train.groupby('Alley').count()

In [ ]:
#次に相関が0.2未満のうち完全に機械学習に関係ないもしくはほかのデータ項目とダブりがあるものを削除する
#まずは定性データから
#heatingの相関は0.2未満であり、HeatingはHeatingQuality（相関が高い）で機械学習に補えると判断し削除
#Functionalityは他のQyalityで補うことのできる横目と判断し、削除（相関も0.2未満）
#AlleyはNone（Alleyがない）ことが多いうえ、相関も0.2未満であることから削除
#PoolQCは相関も低いことから、SaePriceを決定する重要な要素ではないと判断し削除
#ExternalConditionも相関が低く、また「ExternalQual」と似た要素があるため削除
#FenceはSalePriceとの相関が低く、あまり有用でないと考え削除
#LandContourも相関が低く、あまり有用なデータではないと考え削除
#Condition1も相関が低く、機械学習において有用なデータではないと考え削除
#LotConfig, BsmtCond, RoofMatl, BldgTypeについてはほかの項目と関連がありそうなため、いったん保留

del train['Heating']
del train['Functional']
del train['Alley']
del train['PoolQC']
del train['ExterCond']
del train['Fence']
del train['LandContour']
del train['Condition1']

In [ ]:
#テストデータにも同様の処理
del test['Heating']
del test['Functional']
del test['Alley']
del test['PoolQC']
del test['ExterCond']
del test['Fence']
del test['LandContour']
del test['Condition1']

In [ ]:
#定量データを確認する前に、定量データの統計量を確認
train.describe()

In [ ]:
#次に定量データを確認
#BedroomはTotRmsAbvGrdと足したうえで、削除（TotRmsAbvGrdにBedroomsの数が含まれていないため）
#KitchenについてはTotRmsAbvGrdに含まれていると仮定、またSalePriceとの相関も低いことから削除
#EnclosedPorch/screenporchについては、SalePriceとの相関が低いことから削除

train['TotalRoom'] = train['BedroomAbvGr'] + train['TotRmsAbvGrd']

In [ ]:
#テストデータにも同様の処理
test['TotalRoom'] = test['BedroomAbvGr'] + test['TotRmsAbvGrd']

In [ ]:
#いったんTotalRoomが含まれているかを確認
train.describe()

In [ ]:
#TotalRoomが含まれていることを確認したため、データ削除をする
del train['BedroomAbvGr']
del train['TotRmsAbvGrd']
del train['KitchenAbvGr']
del train['EnclosedPorch']
del train['ScreenPorch']

In [ ]:
#テストデータにも同様の処理
del test['BedroomAbvGr']
del test['TotRmsAbvGrd']
del test['KitchenAbvGr']
del test['EnclosedPorch']
del test['ScreenPorch']

In [ ]:
#再度学習データの中身を確認
train.count()

In [ ]:
#LotConfig, BsmtCond, RoofMatl, Condition1, BldgTypeについてはほかの項目と関連がありそうなため、いったん保留中
#まずLotConfigから確認
#LotConfigごとのLotFrontageの統計量
train.groupby('LotConfig')['LotFrontage'].describe()

In [ ]:
#LotConfigごとのLotAreaの統計量
train.groupby('LotConfig')['LotArea'].describe()

In [ ]:
#各LotConfig毎にカテゴライズ化していたところおおよその傾向があることはわかった
#一方でLotConfigを用いなくても、数値の傾向を見ることである程度SalePriceがどの程度になるかを予測できるのではないかと考える
#そのためLotConfigは削除する

del train['LotConfig']

In [ ]:
#テストデータも同様の処理を実施
del test['LotConfig']

In [ ]:
#次にBsmtCondについてみる
#BsmtCondと他の定性データとの相関を確認する
from sklearn.feature_selection import mutual_info_classif

cat_cols = train.select_dtypes(include='object').apply(lambda x: x.astype('category').cat.codes)
mi = mutual_info_classif(cat_cols, train['BsmtCond'].astype('category').cat.codes)
pd.Series(mi, index=cat_cols.columns).sort_values(ascending=False)

In [ ]:
#まずはBasmtCondの中身を確認

train.groupby('BsmtCond').size()

In [ ]:
#関連性のあるBsmtQualも確認
train.groupby('BsmtQual').size()

In [ ]:
#BsmtCondについては出0田に偏りがあるため削除
#一方でBsmtQualについては、データをマッピング化して使用
del train['BsmtCond']
del test['BsmtCond']

In [ ]:
#マッピング化
mapping = {
    'Ex':'High',
    'Gd': 'High',
    'TA':'Fair',
    'Fa':'Fair',
}

train['BsmtQual'] = train['BsmtQual'].replace(mapping)
test['BsmtQual'] = test['BsmtQual'].replace(mapping)

In [ ]:
#次にRoofMatlを確認する
#RoofMatlと他の定性データとの相関を確認する

cat_cols = train.select_dtypes(include='object').apply(lambda x: x.astype('category').cat.codes)
mi = mutual_info_classif(cat_cols, train['RoofMatl'].astype('category').cat.codes)
pd.Series(mi, index=cat_cols.columns).sort_values(ascending=False)

In [ ]:
#RoofMatlはほかの定性データと掛け合わせが厳しい
#SalePriceとの相関（0.2未満）も加味すると、これ以上salepriceを決めるために貢献できる要素がないと考え削除
del train['RoofMatl']
del test['RoofMatl']

In [ ]:
#次にBldgTypeを確認
#BldgTypeと他の定性データとの相関を確認する

cat_cols = train.select_dtypes(include='object').apply(lambda x: x.astype('category').cat.codes)
mi = mutual_info_classif(cat_cols, train['BldgType'].astype('category').cat.codes)
pd.Series(mi, index=cat_cols.columns).sort_values(ascending=False)

In [ ]:
#BldgTypeはほかの定性データと掛け合わせが厳しい
#SalePriceとの相関（0.2未満）も加味すると、これ以上salepriceを決めるために貢献できる要素がないと考え削除
del train['BldgType']
del test['BldgType']

In [ ]:
#あらためて定性データの中身とSalePriceとの相関を確認

def correlation_ratio(categories, values):
    """カテゴリ（定性）と数値（定量）の関係を測る相関比ηを返す"""
    fcat, _ = pd.factorize(categories)
    mean_per_cat = np.array([values[fcat == i].mean() for i in range(len(np.unique(fcat)))])
    n_per_cat = np.array([np.sum(fcat == i) for i in range(len(np.unique(fcat)))])
    overall_mean = values.mean()
    numerator = np.sum(n_per_cat * (mean_per_cat - overall_mean)**2)
    denominator = np.sum((values - overall_mean)**2)
    return np.sqrt(numerator / denominator) if denominator != 0 else 0

# 定性データの列を抽出
cat_cols = train.select_dtypes(include='object').columns

# 目的変数（定量データ）
target = 'SalePrice'

# 各カテゴリ列との相関比を計算
results = []
for col in cat_cols:
    eta = correlation_ratio(train[col], train[target])
    results.append((col, eta))

# 結果をDataFrame化
corr_eta_df = pd.DataFrame(results, columns=['Column', 'Eta']).sort_values(by='Eta', ascending=False)
corr_eta_df

In [ ]:
#現在の定量データの相関を確認
plt.figure(figsize=(40,40))
sns.heatmap(train.corr(numeric_only=True), annot=True)

In [ ]:
#次に相関が0.3未満の項目を見る
#まずは定量データから見る
#BsmtUnfSFはTotalBsmtSFと相関（0.42）ある、この相関図を確かめる

import seaborn as sns
import matplotlib.pyplot as plt

#BsmtUnfSFとTotalBsmtSFの関係を可視化
sns.scatterplot(x='BsmtUnfSF', y='TotalBsmtSF', data=train)

plt.title('Correlationship')
plt.xlabel('BsmtUnfSF')
plt.ylabel('TotalBsmtSF')
plt.show()

In [ ]:
#BsmtUnfSFとTotalBsmtSFを組み合わせて、SalePriceとの相関を見てみる
#今回はTotalBsmtのうち、建設が完了しているところの面積を出す
train['f_BsmtSF'] = train['TotalBsmtSF'] - train['BsmtUnfSF']

In [ ]:
#テストデータの欠損値を確認

test.isnull().sum()

In [ ]:
#test['TotalBsmtSF'] と test['BsmtUnfSF']でNAとなっているところを確認

NA_BSF = test[test['TotalBsmtSF'].isnull()]
NA_BSF

In [ ]:
#上記からBsmtがそもそも存在しないため、NAとなっていると考える
#このデータ以外は通常通り計算し、上記データは「0」とする
test['f_BsmtSF'] = test['TotalBsmtSF'] - test['BsmtUnfSF']
test['f_BsmtSF'] = (test['TotalBsmtSF'] - test['BsmtUnfSF']).fillna(0)

In [ ]:
#SalePriceとの相関を見てみる
corr = train['f_BsmtSF'].corr(train['SalePrice'])
print(f'相関係数: {corr:.3f}')

In [ ]:
#新規で項目を作れたこと、またBsmtUnfSFは削除する
del train['BsmtUnfSF']
del test['BsmtUnfSF']

In [ ]:
#BsmtFullBathを次に見る
#BsmtFullBathとBsmtFinSF1の関係を可視化
#下記グラフから確かに相関はあるものの、BsmtFullBathが特定の数に集約しているために発生している
#相関係数だけみると強い相関はあるが、データ加工に至るものではない

sns.scatterplot(x='BsmtFullBath', y='BsmtFinSF1', data=train)

plt.title('Correlationship')
plt.xlabel('BsmtFullBath')
plt.ylabel('BsmtFinSF1')
plt.show()

In [ ]:
#念のためSalePriceとの相関を可視化する
sns.scatterplot(x='BsmtFullBath', y='SalePrice', data=train)

plt.title('Correlationship')
plt.xlabel('BsmtFullBath')
plt.ylabel('SalePrice')
plt.show()

In [ ]:
#上記グラフから相関係数上は0.23となっているがこれはBsmtfullBathが特定の数に固まっているから（風呂の個数）
#上記グラフでは風呂の数に関係なく、SalePriceにもだいぶバラツキがある
#そのため機械学習に悪影響を及ぼす可能性があるためBsmtFullBathを削除する

del train['BsmtFullBath']
del test['BsmtFullBath']

In [ ]:
#次にHalfBathを見る
#こちらも相関を可視化してみる
#HalfBathと一番相関のある2ndFlrSFを見る

sns.scatterplot(x='HalfBath', y='2ndFlrSF', data=train)

plt.title('Correlationship')
plt.xlabel('HalfBath')
plt.ylabel('2ndFlrSF')
plt.show()

In [ ]:
#念のためSalePriceとの相関を可視化する
sns.scatterplot(x='HalfBath', y='SalePrice', data=train)

plt.title('Correlationship')
plt.xlabel('HalfBath')
plt.ylabel('SalePrice')
plt.show()

In [ ]:
#上記グラフから相BsmtfullBathと同様に特定の数に固まっているから（風呂の個数）
#上記グラフでは風呂の数に関係なく、SalePriceにもだいぶバラツキがある
#そのため機械学習に悪影響を及ぼす可能性があるためHalfBathを削除する

del train['HalfBath']
del test['HalfBath']

In [ ]:
#次に定性データを確認する
#あらためて定性データの中身とSalePriceとの相関を確認

def correlation_ratio(categories, values):
    """カテゴリ（定性）と数値（定量）の関係を測る相関比ηを返す"""
    fcat, _ = pd.factorize(categories)
    mean_per_cat = np.array([values[fcat == i].mean() for i in range(len(np.unique(fcat)))])
    n_per_cat = np.array([np.sum(fcat == i) for i in range(len(np.unique(fcat)))])
    overall_mean = values.mean()
    numerator = np.sum(n_per_cat * (mean_per_cat - overall_mean)**2)
    denominator = np.sum((values - overall_mean)**2)
    return np.sqrt(numerator / denominator) if denominator != 0 else 0

# 定性データの列を抽出
cat_cols = train.select_dtypes(include='object').columns

# 目的変数（定量データ）
target = 'SalePrice'

# 各カテゴリ列との相関比を計算
results = []
for col in cat_cols:
    eta = correlation_ratio(train[col], train[target])
    results.append((col, eta))

# 結果をDataFrame化
corr_eta_df = pd.DataFrame(results, columns=['Column', 'Eta']).sort_values(by='Eta', ascending=False)
corr_eta_df

In [ ]:
#まずPavedDriveを確認する
#PavedDriveの項目の分布を確認する
#以下のように、かなり分布が偏っている
train.groupby('PavedDrive').size()

In [ ]:
#念のためPavedDriveと他の定性データとの相関を確認する

cat_cols = train.select_dtypes(include='object').apply(lambda x: x.astype('category').cat.codes)
mi = mutual_info_classif(cat_cols, train['PavedDrive'].astype('category').cat.codes)
pd.Series(mi, index=cat_cols.columns).sort_values(ascending=False)

In [ ]:
#分布がかなりかたよっているうえ、ほかの定性データとの組み合わせや相関もないことから、これ以上SalePriceのデータ学習に高めるための構図は厳しい
#そのため本データ項目は削除する

del train['PavedDrive']
del test['PavedDrive']

In [ ]:
#次にRoofStyleを確認
#まずは中身を確認
train.groupby('RoofStyle').size()

In [ ]:
#念のためPavedDriveと他の定性データとの相関を確認する

cat_cols = train.select_dtypes(include='object').apply(lambda x: x.astype('category').cat.codes)
mi = mutual_info_classif(cat_cols, train['RoofStyle'].astype('category').cat.codes)
pd.Series(mi, index=cat_cols.columns).sort_values(ascending=False)

In [ ]:
#分布がかなりかたよっているうえ、ほかの定性データとの組み合わせや相関もないことから、これ以上SalePriceのデータ学習に高めるための構図は厳しい
#そのため本データ項目は削除する

del train['RoofStyle']
del test['RoofStyle']

In [ ]:
#次にElectricalを確認
#まずは中身を確認
train.groupby('Electrical').size()

In [ ]:
#念のためElectricalと他の定性データとの相関を確認する

cat_cols = train.select_dtypes(include='object').apply(lambda x: x.astype('category').cat.codes)
mi = mutual_info_classif(cat_cols, train['Electrical'].astype('category').cat.codes)
pd.Series(mi, index=cat_cols.columns).sort_values(ascending=False)

In [ ]:
#分布がかなりかたよっているうえ、ほかの定性データとの組み合わせや相関もないことから、これ以上SalePriceのデータ学習に高めるための構図は厳しい
#そのため本データ項目は削除する

del train['Electrical']
del test['Electrical']

In [ ]:
#次にCentralAirを確認
#まずは中身を確認
train.groupby('CentralAir').size()

In [ ]:
#念のためCentralAirと他の定性データとの相関を確認する

cat_cols = train.select_dtypes(include='object').apply(lambda x: x.astype('category').cat.codes)
mi = mutual_info_classif(cat_cols, train['CentralAir'].astype('category').cat.codes)
pd.Series(mi, index=cat_cols.columns).sort_values(ascending=False)

In [ ]:
#分布がかなりかたよっているうえ、ほかの定性データとの組み合わせや相関もないことから、これ以上SalePriceのデータ学習に高めるための構図は厳しい
#そのため本データ項目は削除する

del train['CentralAir']
del test['CentralAir']

In [ ]:
#次にLotShapeを確認
#まずは中身を確認
train.groupby('LotShape').size()

In [ ]:
#念のためLotShapeと他の定性データとの相関を確認する

cat_cols = train.select_dtypes(include='object').apply(lambda x: x.astype('category').cat.codes)
mi = mutual_info_classif(cat_cols, train['LotShape'].astype('category').cat.codes)
pd.Series(mi, index=cat_cols.columns).sort_values(ascending=False)

In [ ]:
#LotShapeのうち、Iregularがつく項目はすべてIregularとまとめた場合の相関を確認する
threshold = 500

freq = train['LotShape'].value_counts()
rare = freq[freq < threshold].index

train['LotShape_grouped'] = train['LotShape'].replace(rare, 'Other')
test['LotShape_grouped'] = test['LotShape'].replace(rare, 'Other')

In [ ]:
#念のため再度内容を確認
train.groupby('LotShape_grouped').size()

In [ ]:
#グルーピング後の相関を再度確認
def correlation_ratio(categories, values):
    # NaN除外
    mask = ~pd.isna(categories) & ~pd.isna(values)
    categories = categories[mask]
    values = values[mask]
    
    # カテゴリごとの平均
    cat_means = values.groupby(categories).mean()
    # 全体平均
    overall_mean = values.mean()
    # 各カテゴリの件数
    n = values.groupby(categories).count()
    
    # 分散比
    numerator = sum(n * (cat_means - overall_mean)**2)
    denominator = sum((values - overall_mean)**2)
    
    return np.sqrt(numerator / denominator)

eta = correlation_ratio(train['LotShape_grouped'], train['SalePrice'])
print(f'相関比 (η) = {eta:.3f}')

In [ ]:
#グルーピング前後で相関が下がっているため、SalePriceとの相関はこれ以上改善の余地がないと判断
#機械学習に有用でないと考え削除

del train['LotShape']
del train['LotShape_grouped']
del test['LotShape']
del test['LotShape_grouped']

In [ ]:
#あらためて定性データの中身とSalePriceとの相関を確認

def correlation_ratio(categories, values):
    """カテゴリ（定性）と数値（定量）の関係を測る相関比ηを返す"""
    fcat, _ = pd.factorize(categories)
    mean_per_cat = np.array([values[fcat == i].mean() for i in range(len(np.unique(fcat)))])
    n_per_cat = np.array([np.sum(fcat == i) for i in range(len(np.unique(fcat)))])
    overall_mean = values.mean()
    numerator = np.sum(n_per_cat * (mean_per_cat - overall_mean)**2)
    denominator = np.sum((values - overall_mean)**2)
    return np.sqrt(numerator / denominator) if denominator != 0 else 0

# 定性データの列を抽出
cat_cols = train.select_dtypes(include='object').columns

# 目的変数（定量データ）
target = 'SalePrice'

# 各カテゴリ列との相関比を計算
results = []
for col in cat_cols:
    eta = correlation_ratio(train[col], train[target])
    results.append((col, eta))

# 結果をDataFrame化
corr_eta_df = pd.DataFrame(results, columns=['Column', 'Eta']).sort_values(by='Eta', ascending=False)
corr_eta_df

In [ ]:
#次にGarageCondとGarageQualを確認
#この２つは似た要素があると考えられるため、２つの項目の相関を確認

from scipy.stats import chi2_contingency
import numpy as np
import pandas as pd

def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    chi2 = chi2_contingency(ct)[0]
    n = ct.sum().sum()
    r, k = ct.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1)))

v = cramers_v(train['GarageCond'], train['GarageQual'])
print(f"Cramér's V (カテゴリ同士の関連度): {v:.3f}")

In [ ]:
#各項目のデータ分布が気になるため、確認
train.groupby('GarageCond').size()

In [ ]:
#各項目のデータ分布が気になるため、確認
train.groupby('GarageQual').size()

In [ ]:
#GarageQualとGarageCondの分布が1つの項目にかたよっていることから、そもそも機械学習に貢献できる可能性が少ない
#この2つの項目については削除する

del train['GarageCond']
del train['GarageQual']
del test['GarageCond']
del test['GarageQual']

In [ ]:
#次にHouseStyleを確認
#各項目のデータ分布が気になるため、確認
train.groupby('HouseStyle').size()

In [ ]:
#HousestyleごとのSalepriceも確認
#はっきりとした傾向があるかを見てみる
train.groupby('HouseStyle')['SalePrice'].describe()

In [ ]:
#HouseStyleと他の定性データとの相関を確認する

cat_cols = train.select_dtypes(include='object').apply(lambda x: x.astype('category').cat.codes)
mi = mutual_info_classif(cat_cols, train['HouseStyle'].astype('category').cat.codes)
pd.Series(mi, index=cat_cols.columns).sort_values(ascending=False)

In [ ]:
#念のため定量データとの相関も見る
def correlation_ratio(categories, values):
    """
    相関比 η（定性 × 定量）
    """
    mask = ~pd.isna(categories) & ~pd.isna(values)
    categories = categories[mask]
    values = values[mask]
    
    cat_means = values.groupby(categories).mean()
    overall_mean = values.mean()
    n = values.groupby(categories).count()
    
    numerator = sum(n * (cat_means - overall_mean)**2)
    denominator = sum((values - overall_mean)**2)
    
    return np.sqrt(numerator / denominator) if denominator != 0 else np.nan

#HouseStyleと他の全数値列の相関を調べる
cat_col = 'HouseStyle'

# 数値列だけを抽出
num_cols = train.select_dtypes(include=[np.number]).columns

results = []
for col in num_cols:
    eta = correlation_ratio(train[cat_col], train[col])
    results.append((col, eta))

# 結果をDataFrame化＆ソート
corr_eta_df = pd.DataFrame(results, columns=['Numeric_Variable', 'Eta'])
corr_eta_df.sort_values(by='Eta', ascending=False, inplace=True)

print(corr_eta_df)

In [ ]:
#上記相関を確認するとHousestyleは2ndFlrSFやGrLivAreaと相関が高いことが伺える
#これはHousestyleがその建物の構造を示しており、構造と実際の居住面積や床面積が大きくなることは必然である
#また定量データである2ndFlrSFやGrLivAreaがあれば、HouseStyleの代わりになると考えられる
#そのためHouseStyleは削除する

del train['HouseStyle']
del test['HouseStyle']

In [ ]:
#あらためて定性データの中身とSalePriceとの相関を確認

def correlation_ratio(categories, values):
    """カテゴリ（定性）と数値（定量）の関係を測る相関比ηを返す"""
    fcat, _ = pd.factorize(categories)
    mean_per_cat = np.array([values[fcat == i].mean() for i in range(len(np.unique(fcat)))])
    n_per_cat = np.array([np.sum(fcat == i) for i in range(len(np.unique(fcat)))])
    overall_mean = values.mean()
    numerator = np.sum(n_per_cat * (mean_per_cat - overall_mean)**2)
    denominator = np.sum((values - overall_mean)**2)
    return np.sqrt(numerator / denominator) if denominator != 0 else 0

# 定性データの列を抽出
cat_cols = train.select_dtypes(include='object').columns

# 目的変数（定量データ）
target = 'SalePrice'

# 各カテゴリ列との相関比を計算
results = []
for col in cat_cols:
    eta = correlation_ratio(train[col], train[target])
    results.append((col, eta))

# 結果をDataFrame化
corr_eta_df = pd.DataFrame(results, columns=['Column', 'Eta']).sort_values(by='Eta', ascending=False)
corr_eta_df

In [ ]:
#現在の定量データの相関を確認
plt.figure(figsize=(40,40))
sns.heatmap(train.corr(numeric_only=True), annot=True)

In [ ]:
#次に定量データのうち、SalePriceとの相関が0.3未満のものを探る
#まずはLotArea
#統計量を確認

train['LotArea'].describe()

In [ ]:
#LotAreaとSalePriceの関係をヒストグラムで確認
plt.figure(figsize=(8,5))
plt.scatter(train['LotArea'], train['SalePrice'], alpha=0.5)
plt.xlabel('LotArea')
plt.ylabel('SalePrice')
plt.title('LotArea vs SalePrice')
plt.show()

In [ ]:
#外れ値がいくつかあるので、外れ値を外した後のデータセットを作成
def remove_outliers(df, column, k=1.5):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - k * IQR
    upper = Q3 + k * IQR
    return df[(df[column] >= lower) & (df[column] <= upper)]

train_no_out = remove_outliers(train, 'LotArea')

plt.figure(figsize=(8,5))
plt.scatter(train_no_out['LotArea'], train_no_out['SalePrice'], alpha=0.5)
plt.xlabel('LotArea')
plt.ylabel('SalePrice')
plt.title('LotArea vs SalePrice (Outliers Removed)')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(train['LotFrontage'], train['SalePrice'], alpha=0.5)
plt.xlabel('LotFrontage')
plt.ylabel('SalePrice')
plt.title('LotFrontage vs SalePrice')
plt.show()

In [ ]:
train_no_out_LF = remove_outliers(train, 'LotFrontage')

plt.figure(figsize=(8,5))
plt.scatter(train_no_out_LF['LotFrontage'], train_no_out_LF['SalePrice'], alpha=0.5)
plt.xlabel('LotArea')
plt.ylabel('SalePrice')
plt.title('LotArea vs SalePrice (Outliers Removed)')
plt.show()

In [ ]:
#LotFrontage, LotAreaを削除する
del train['LotArea']
del train['LotFrontage']
del test['LotArea']
del test['LotFrontage']

In [ ]:
#YearBuiltやYearRemodAddと組み合わせると、経年劣化の度合を示すことができるのではないかと考える
#いったんGarageYrBltについてはこのままとする
#一方でGarageCarsやGarageAreasは、1stFlrSFや2ndFlrSf、TotalBsmtSFと組み合わせると、居住地全体の面積を示すことがでいる可能性あり
#GarageCarsとGarageAreasは付加価値的存在になりうる？（ガレージがあるかどうか、広さ）
#そのため居住地は居住地の広さで、Garage関連はGarageで独立して示す
#まずは居住区域の広さから、新規項目の「House_SF」を作成する
#居住区域の広さを算出するために使った項目（TotalBsmtSF, 1stFlrSF, 2ndFlrSF）は削除する
#またGrLivAreaについても居住区域の広さ（地上階）を示すため、ダブりをなくすためにも削除する
#テストデータ分のTotalBsmtSFについては1件欠損値となっていたため、こちらは「0」となおす

test['TotalBsmtSF'] = test['TotalBsmtSF'].fillna(0)

train['House_SF'] = train['TotalBsmtSF'] + train['1stFlrSF'] + train['2ndFlrSF']
test['House_SF'] = test['TotalBsmtSF'] + test['1stFlrSF'] + test['2ndFlrSF']

del train['TotalBsmtSF']
del train['1stFlrSF']
del train['2ndFlrSF']
del train['GrLivArea']
del test['TotalBsmtSF']
del test['1stFlrSF']
del test['2ndFlrSF']
del test['GrLivArea']

In [ ]:
#House_SFと他の項目の相関を確認
corr = train.corr(numeric_only=True)
corr['House_SF'].sort_values(ascending=False)

In [ ]:
#現在の相関を確認する
plt.figure(figsize=(20,20))
sns.heatmap(train.corr(numeric_only=True), annot=True)

In [ ]:
#あらためて定性データの中身とSalePriceとの相関を確認

def correlation_ratio(categories, values):
    """カテゴリ（定性）と数値（定量）の関係を測る相関比ηを返す"""
    fcat, _ = pd.factorize(categories)
    mean_per_cat = np.array([values[fcat == i].mean() for i in range(len(np.unique(fcat)))])
    n_per_cat = np.array([np.sum(fcat == i) for i in range(len(np.unique(fcat)))])
    overall_mean = values.mean()
    numerator = np.sum(n_per_cat * (mean_per_cat - overall_mean)**2)
    denominator = np.sum((values - overall_mean)**2)
    return np.sqrt(numerator / denominator) if denominator != 0 else 0

# 定性データの列を抽出
cat_cols = train.select_dtypes(include='object').columns

# 目的変数（定量データ）
target = 'SalePrice'

# 各カテゴリ列との相関比を計算
results = []
for col in cat_cols:
    eta = correlation_ratio(train[col], train[target])
    results.append((col, eta))

# 結果をDataFrame化
corr_eta_df = pd.DataFrame(results, columns=['Column', 'Eta']).sort_values(by='Eta', ascending=False)
corr_eta_df

In [ ]:
#次にSalePriceとの相関が0.4未満の定性データを確認する
#まずはMSZoning
#データの中身を確認

train.groupby('MSZoning').size()

In [ ]:
#データに偏りがあることから、過学習を避けるためにも削除する
del train['MSZoning']
del test['MSZoning']

In [ ]:
#次にBsmtExposureを確認
#データの中身を確認
train.groupby('BsmtExposure').size()

In [ ]:
#念のため定量データとの相関も見る
def correlation_ratio(categories, values):
    """
    相関比 η（定性 × 定量）
    """
    mask = ~pd.isna(categories) & ~pd.isna(values)
    categories = categories[mask]
    values = values[mask]
    
    cat_means = values.groupby(categories).mean()
    overall_mean = values.mean()
    n = values.groupby(categories).count()
    
    numerator = sum(n * (cat_means - overall_mean)**2)
    denominator = sum((values - overall_mean)**2)
    
    return np.sqrt(numerator / denominator) if denominator != 0 else np.nan

#BsmtExposureと他の全数値列の相関を調べる
cat_col = 'BsmtExposure'

# 数値列だけを抽出
num_cols = train.select_dtypes(include=[np.number]).columns

results = []
for col in num_cols:
    eta = correlation_ratio(train[cat_col], train[col])
    results.append((col, eta))

# 結果をDataFrame化＆ソート
corr_eta_df = pd.DataFrame(results, columns=['Numeric_Variable', 'Eta'])
corr_eta_df.sort_values(by='Eta', ascending=False, inplace=True)

print(corr_eta_df)

In [ ]:
#他の定性データの相関も確認
cat_cols = train.select_dtypes(include='object').apply(lambda x: x.astype('category').cat.codes)
mi = mutual_info_classif(cat_cols, train['BsmtExposure'].astype('category').cat.codes)
pd.Series(mi, index=cat_cols.columns).sort_values(ascending=False)

In [ ]:
#BsmtFinSF1はf_BsmtSFの一部であること から、BsmtFinSF1は削除する
del train['BsmtFinSF1']
del train['BsmtFinType1']
del train['BsmtExposure']
del test['BsmtFinSF1']
del test['BsmtFinType1']
del test['BsmtExposure']

In [ ]:
#次にwoodeckとopenporchを確認
#この2つの要素は「玄関付近の付加価値要素」として映っていると考えられる
#そのため２つの面積を合わせたうえで、相関を作ることも1つの手になる

train['entrance'] = train['WoodDeckSF'] + train['OpenPorchSF']
test['entrance'] = test['WoodDeckSF'] + test['OpenPorchSF']

In [ ]:
#Entranceに使ったWoodDeckとOpenPorchは削除する
del train['WoodDeckSF']
del train['OpenPorchSF']
del test['WoodDeckSF']
del test['OpenPorchSF']

In [ ]:
#現在の相関を確認する
plt.figure(figsize=(20,20))
sns.heatmap(train.corr(numeric_only=True), annot=True)

In [ ]:
#現在の学習データの中身を確認
train.head()

In [ ]:
#あらためて定性データの中身とSalePriceとの相関を確認

def correlation_ratio(categories, values):
    """カテゴリ（定性）と数値（定量）の関係を測る相関比ηを返す"""
    fcat, _ = pd.factorize(categories)
    mean_per_cat = np.array([values[fcat == i].mean() for i in range(len(np.unique(fcat)))])
    n_per_cat = np.array([np.sum(fcat == i) for i in range(len(np.unique(fcat)))])
    overall_mean = values.mean()
    numerator = np.sum(n_per_cat * (mean_per_cat - overall_mean)**2)
    denominator = np.sum((values - overall_mean)**2)
    return np.sqrt(numerator / denominator) if denominator != 0 else 0

# 定性データの列を抽出
cat_cols = train.select_dtypes(include='object').columns

# 目的変数（定量データ）
target = 'SalePrice'

# 各カテゴリ列との相関比を計算
results = []
for col in cat_cols:
    eta = correlation_ratio(train[col], train[target])
    results.append((col, eta))

# 結果をDataFrame化
corr_eta_df = pd.DataFrame(results, columns=['Column', 'Eta']).sort_values(by='Eta', ascending=False)
corr_eta_df

In [ ]:
#次にSaleConditionについて確認
#データ項目の数を確認
train.groupby('SaleType').size()

In [ ]:
#同じようにSaleConditionについても確認
train.groupby('SaleCondition').size()

In [ ]:
#どちらもデータが偏っていることから、過学習を避けるため削除する

del train['SaleCondition']
del train['SaleType']
del test['SaleCondition']
del test['SaleType']

In [ ]:
#続いてExterior1st
#こちらも最初にデータの中身を確認

train.groupby('Exterior1st').size()

In [ ]:
#Exterior2ndも同様にデータの中身を確認

train.groupby('Exterior2nd').size()

In [ ]:
#Exterior関連の項目は希少データと過大で0田が混合しているため、ノイズを避けるためにも削除する
del train['Exterior1st']
del train['Exterior2nd']
del test['Exterior1st']
del test['Exterior2nd']

In [ ]:
#現在の学習データを確認
train.head()

In [ ]:
#現在の相関を確認する
plt.figure(figsize=(20,20))
sns.heatmap(train.corr(numeric_only=True), annot=True)

In [ ]:
#次に相関が0.5未満の項目を確認する
#まずはMasVnrTypeから
#データの分布を確認し、特定の項目に偏りがないかを確認する

train.groupby('MasVnrType').size()

In [ ]:
#定量データのMasVnrAreaと関連があり、Type別の統計量を確認してみる

train.groupby('MasVnrType')['MasVnrArea'].describe()

In [ ]:
#MasVnrTypeが割り振られているにも関わらず、0となるケースには違和感あり
#TypeがNone以外のうち、0となっている項目を確認する

mask = (train['MasVnrType'] != 'None') & (train['MasVnrArea'] == 0)
train[mask]

In [ ]:
#テストデータも念のため同様の処理を実施
mask_t = (test['MasVnrType'] != 'None') & (test['MasVnrArea'] == 0)
test[mask_t]

In [ ]:
#上記データについては、データの入力ミスがあったと仮定する
#MasVnrAreaについては中央値で割り振る

train.loc[mask, 'MasVnrArea'] = train.groupby('MasVnrType')['MasVnrArea'].transform('median')
test.loc[mask_t, 'MasVnrArea'] = test.groupby('MasVnrType')['MasVnrArea'].transform('median')

In [ ]:
#再度MasVnrType別のMasVnrAreaを確認する

train.groupby('MasVnrType')['MasVnrArea'].describe()

In [ ]:
#タイプ別に中央値との差を確認し、その項目を新規のデータ項目として追加してみる
median_by_type = train.groupby('MasVnrType')['MasVnrArea'].transform('median')
train['MasVnrArea_ratio_to_type'] = train['MasVnrArea'] / median_by_type

In [ ]:
#現在の相関を確認する
plt.figure(figsize=(20,20))
sns.heatmap(train.corr(numeric_only=True), annot=True)

In [ ]:
#元のデータ項目（MasVnrTypeやMasVnrArea）の相関よりも下がっていることから、新規データ項目を採用しない
#MasVnr関連の項目は独立させて機械学習に用いる
del train['MasVnrArea_ratio_to_type']

In [ ]:
#続いてGarageType
#こちらもまずはデータに偏りがないかを確認

train.groupby('GarageType').size()

In [ ]:
#Attachdとdetachdの２つにデータが集中しているため、ほかの項目はOtherでまとめる
value_counts = train['GarageType'].value_counts()
major = value_counts.index[:2]     # 上位2つ
train['GarageType'] = train['GarageType'].where(train['GarageType'].isin(major), 'Other')

In [ ]:
#テストデータも同様に
value_counts_t = test['GarageType'].value_counts()
major = value_counts.index[:2]     # 上位2つ
test['GarageType'] = test['GarageType'].where(test['GarageType'].isin(major), 'Other')

In [ ]:
#GarageTypeはGarage関連の項目（GarageArea, GarageCars）と関連があると見込む

def correlation_ratio(categories, values):
    """カテゴリ（定性）と数値（定量）の関係を測る相関比ηを返す"""
    fcat, _ = pd.factorize(categories)
    mean_per_cat = np.array([values[fcat == i].mean() for i in range(len(np.unique(fcat)))])
    n_per_cat = np.array([np.sum(fcat == i) for i in range(len(np.unique(fcat)))])
    overall_mean = values.mean()
    numerator = np.sum(n_per_cat * (mean_per_cat - overall_mean)**2)
    denominator = np.sum((values - overall_mean)**2)
    return np.sqrt(numerator / denominator) if denominator != 0 else 0

# 定性データの列を抽出
cat_cols = train.select_dtypes(include='object').columns

# 目的変数（定量データ）
target = 'GarageCars'

# 各カテゴリ列との相関比を計算
results = []
for col in cat_cols:
    eta = correlation_ratio(train[col], train[target])
    results.append((col, eta))

# 結果をDataFrame化
corr_eta_df = pd.DataFrame(results, columns=['Column', 'Eta']).sort_values(by='Eta', ascending=False)
corr_eta_df

In [ ]:
#GarageTypeとGarageCars自体に直接の因果関係はないものの、詳細は見る必要あり
#GarageTypeごとのGarageCarsとSalePriceの推移を確認すべく、可視化する
#以下の箱ひげ図から車の収容量が4台を除き、基本的にはDetached→Attatched→Othersの順でSalePriceが高くなる傾向にある

plt.figure(figsize=(12,6))
sns.boxplot(data=train, x='GarageCars', y='SalePrice', hue='GarageType')
plt.title('SalePrice by GarageCars and GarageType')
plt.xlabel('Number of Garage Cars')
plt.ylabel('Sale Price')
plt.legend(title='Garage Type')
plt.show()

In [ ]:
#GarageTypeをダミー変数化して、機械学習を実施できる内容にする
train = pd.get_dummies(train, columns=['GarageType'], drop_first=True)
test = pd.get_dummies(test, columns=['GarageType'], drop_first=True)

In [ ]:
#あらためて定性データの中身とSalePriceとの相関を確認

def correlation_ratio(categories, values):
    """カテゴリ（定性）と数値（定量）の関係を測る相関比ηを返す"""
    fcat, _ = pd.factorize(categories)
    mean_per_cat = np.array([values[fcat == i].mean() for i in range(len(np.unique(fcat)))])
    n_per_cat = np.array([np.sum(fcat == i) for i in range(len(np.unique(fcat)))])
    overall_mean = values.mean()
    numerator = np.sum(n_per_cat * (mean_per_cat - overall_mean)**2)
    denominator = np.sum((values - overall_mean)**2)
    return np.sqrt(numerator / denominator) if denominator != 0 else 0

# 定性データの列を抽出
cat_cols = train.select_dtypes(include='object').columns

# 目的変数（定量データ）
target = 'SalePrice'

# 各カテゴリ列との相関比を計算
results = []
for col in cat_cols:
    eta = correlation_ratio(train[col], train[target])
    results.append((col, eta))

# 結果をDataFrame化
corr_eta_df = pd.DataFrame(results, columns=['Column', 'Eta']).sort_values(by='Eta', ascending=False)
corr_eta_df

In [ ]:
#次にQuality関連の項目を見る（HeatingQC, Bsmt_Condition_Quality, FireplaceQu, KitchenQual, ExterQual)
#この項目はOverallQualに関連している可能性が高い
#相関を確認する

def correlation_ratio(categories, values):
    """カテゴリ（定性）と数値（定量）の関係を測る相関比ηを返す"""
    fcat, _ = pd.factorize(categories)
    mean_per_cat = np.array([values[fcat == i].mean() for i in range(len(np.unique(fcat)))])
    n_per_cat = np.array([np.sum(fcat == i) for i in range(len(np.unique(fcat)))])
    overall_mean = values.mean()
    numerator = np.sum(n_per_cat * (mean_per_cat - overall_mean)**2)
    denominator = np.sum((values - overall_mean)**2)
    return np.sqrt(numerator / denominator) if denominator != 0 else 0

# 定性データの列を抽出
cat_cols = train.select_dtypes(include='object').columns

# 目的変数（定量データ）
target = 'OverallQual'

# 各カテゴリ列との相関比を計算
results = []
for col in cat_cols:
    eta = correlation_ratio(train[col], train[target])
    results.append((col, eta))

# 結果をDataFrame化
corr_eta_df = pd.DataFrame(results, columns=['Column', 'Eta']).sort_values(by='Eta', ascending=False)
corr_eta_df

In [ ]:
#各データ項目の分布も念のため確認
train.groupby('ExterQual').size()

In [ ]:
#ExternalQualについては「High」（Ex, Gd)と「Fair」(TA,Fa)に分けられると考え、まとめる
#そのうえで各ランクを数値化して、機械学習ができるようにする
mapping = {
    'Ex': 'High',
    'Gd': 'High',
    'TA': 'Fair',
    'Fa': 'Fair'
}
train['ExterQual'] = train['ExterQual'].replace(mapping)

order = {'Fair':1, 'High':2}
train['ExterQual_score'] = train['ExterQual'].map(order)

In [ ]:
#テストデータも同様に
mapping = {
    'Ex': 'High',
    'Gd': 'High',
    'TA': 'Fair',
    'Fa': 'Fair'
}
test['ExterQual'] = test['ExterQual'].replace(mapping)

order = {'Fair':1, 'High':2}
test['ExterQual_score'] = test['ExterQual'].map(order)

In [ ]:
#KitchenQualのデータ分布を確認
train.groupby('KitchenQual').size()

In [ ]:
#KitchenQualも、ExternalQualと同様に「High」（Ex, Gd)と「Fair」(TA,Fa)に分けられると考え、まとめる
#そのうえで各ランクを数値化して、機械学習ができるようにする
mapping = {
    'Ex': 'High',
    'Gd': 'High',
    'TA': 'Fair',
    'Fa': 'Fair'
}
train['KitchenQual'] = train['KitchenQual'].replace(mapping)

order = {'Fair':1, 'High':2}
train['KitchenQual_score'] = train['KitchenQual'].map(order)

In [ ]:
#FireplaceQuのデータ分布を確認
train.groupby('FireplaceQu').size()

In [ ]:
#現時点のテストデータの欠損値を確認
test.isnull().sum()

In [ ]:
#KitchenQualにて1件欠損値が出ているため、確認

K_NA = test[test['KitchenQual'].isnull()]
K_NA

In [ ]:
#KitchenがNAとなっているのはおそらくデータ入力の見落としの可能性が高いため、最頻値で穴埋めする
test.groupby('KitchenQual').size()

In [ ]:
#TAで穴埋めする
test['KitchenQual'] = test['KitchenQual'].fillna('TA')

In [ ]:
#テストデータも同様に
mapping = {
    'Ex': 'High',
    'Gd': 'High',
    'TA': 'Fair',
    'Fa': 'Fair'
}
test['KitchenQual'] = test['KitchenQual'].replace(mapping)

order = {'Fair':1, 'High':2}
test['KitchenQual_score'] = test['KitchenQual'].map(order)

In [ ]:
#暖炉があっても分布がバラバラ出るため、データ項目数が少ない項目はまとめる
#そのうえで各ランクを数値化して、機械学習ができるようにする
mapping = {
    'Ex':'High',
    'Gd':'High',
    'TA':'Middle',
    'Fa':'Low',
    'Po':'Low',
    'None':'None'
}
train['FireplaceQu'] = train['FireplaceQu'].replace(mapping)

order = {'None':0, 'Low':1, 'Middle':2, 'High':3}
train['FireplaceQu_score'] = train['FireplaceQu'].map(order)

In [ ]:
#テストデータも同様に
mapping = {
    'Ex':'High',
    'Gd':'High',
    'TA':'Middle',
    'Fa':'Low',
    'Po':'Low',
    'None':'None'
}
test['FireplaceQu'] = test['FireplaceQu'].replace(mapping)

order = {'None':0, 'Low':1, 'Middle':2, 'High':3}
test['FireplaceQu_score'] = test['FireplaceQu'].map(order)

In [ ]:
#当初のデータであるFireplaceは削除する
del train['Fireplaces']
del test['Fireplaces']

In [ ]:
#HeatingQuのデータ分布を確認
train.groupby('HeatingQC').size()

In [ ]:
#HeatingQCも分布がバラバラ出るため、データ項目数が少ない項目はまとめる
#そのうえで各ランクを数値化して、機械学習ができるようにする
#今回分析の安定化のため、ExのみをHighとし、GdはMiddleにする
mapping = {
    'Ex':'High',
    'Gd':'Middle',
    'TA':'Middle',
    'Fa':'Low',
    'Po':'Low',
}
train['HeatingQC'] = train['HeatingQC'].replace(mapping)

order = {'Low':1, 'Middle':2, 'High':3}
train['HeatingQC_score'] = train['HeatingQC'].map(order)

In [ ]:
#テストデータも同様に
mapping = {
    'Ex':'High',
    'Gd':'Middle',
    'TA':'Middle',
    'Fa':'Low',
    'Po':'Low',
}
test['HeatingQC'] = test['HeatingQC'].replace(mapping)

order = {'Low':1, 'Middle':2, 'High':3}
test['HeatingQC_score'] = test['HeatingQC'].map(order)

In [ ]:
#現在の相関を確認する
plt.figure(figsize=(20,20))
sns.heatmap(train.corr(numeric_only=True), annot=True)

In [ ]:
#ここまでの処理で使った元の項目を削除する
del train['HeatingQC']
del train['FireplaceQu']
del train['KitchenQual']
del train['ExterQual']

In [ ]:
#テストデータにも同様の処理を実施
del test['HeatingQC']
del test['FireplaceQu']
del test['KitchenQual']
del test['ExterQual']

In [ ]:
def correlation_ratio(categories, values):
    """カテゴリ（定性）と数値（定量）の関係を測る相関比ηを返す"""
    fcat, _ = pd.factorize(categories)
    mean_per_cat = np.array([values[fcat == i].mean() for i in range(len(np.unique(fcat)))])
    n_per_cat = np.array([np.sum(fcat == i) for i in range(len(np.unique(fcat)))])
    overall_mean = values.mean()
    numerator = np.sum(n_per_cat * (mean_per_cat - overall_mean)**2)
    denominator = np.sum((values - overall_mean)**2)
    return np.sqrt(numerator / denominator) if denominator != 0 else 0

# 定性データの列を抽出
cat_cols = train.select_dtypes(include='object').columns

# 目的変数（定量データ）
target = 'SalePrice'

# 各カテゴリ列との相関比を計算
results = []
for col in cat_cols:
    eta = correlation_ratio(train[col], train[target])
    results.append((col, eta))

# 結果をDataFrame化
corr_eta_df = pd.DataFrame(results, columns=['Column', 'Eta']).sort_values(by='Eta', ascending=False)
corr_eta_df

In [ ]:
#定性データの項目を機械学習できるように定量データに直す
#再度MasVnrTypeを確認

train.groupby('MasVnrType').size()

In [ ]:
#少数データであるBrkCmnについては同じBrk系であるBrkFaceとまとめる

mapping = {
    "BrkCmn":"Brk",
    "BrkFace":'Brk',
    'None':'None',
    'Stone':'Stone',
}

train['MasVnrType'] = train['MasVnrType'].replace(mapping)

In [ ]:
#テストデータも同様に
mapping = {
    "BrkCmn":"Brk",
    "BrkFace":'Brk',
    'None':'None',
    'Stone':'Stone',
}

test['MasVnrType'] = test['MasVnrType'].replace(mapping)

In [ ]:
#MasVnrTypeはMasVnrAreaと関連があると見込んでいるが、実際の相関を確認
def correlation_ratio(categories, values):
    """カテゴリ（定性）と数値（定量）の関係を測る相関比ηを返す"""
    fcat, _ = pd.factorize(categories)
    mean_per_cat = np.array([values[fcat == i].mean() for i in range(len(np.unique(fcat)))])
    n_per_cat = np.array([np.sum(fcat == i) for i in range(len(np.unique(fcat)))])
    overall_mean = values.mean()
    numerator = np.sum(n_per_cat * (mean_per_cat - overall_mean)**2)
    denominator = np.sum((values - overall_mean)**2)
    return np.sqrt(numerator / denominator) if denominator != 0 else 0

# 定性データの列を抽出
cat_cols = train.select_dtypes(include='object').columns

# 目的変数（定量データ）
target = 'MasVnrArea'

# 各カテゴリ列との相関比を計算
results = []
for col in cat_cols:
    eta = correlation_ratio(train[col], train[target])
    results.append((col, eta))

# 結果をDataFrame化
corr_eta_df = pd.DataFrame(results, columns=['Column', 'Eta']).sort_values(by='Eta', ascending=False)
corr_eta_df

In [ ]:
#MasVnrTypeとMasVnrAreaとの間に強い相関（0.68）が確認されるため、この２つで新しい定量項目を作ってみる
#MasVnrType別のAreaの面積を見てみる

train.groupby('MasVnrType')['MasVnrArea'].describe()

In [ ]:
# MasVnrTypeごとのMasVnrAreaの分布を確認

plt.figure(figsize=(12,6))
sns.boxplot(data=train, x='MasVnrType', y='MasVnrArea')
plt.title('Distribution of MasVnrArea by MasVnrType')
plt.xlabel('MasVnrType')
plt.ylabel('MasVnrArea')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# MasVnrTypeごとのSalepriceの確認も分布
#全体的な分布としてNone→Brk→Stoneの順でSalePriceが高くなる
#このMasVnrTypeをダミー変数化して、機械学習に用いるようにする

plt.figure(figsize=(12,6))
sns.boxplot(data=train, x='MasVnrType', y='SalePrice')
plt.title('Distribution of SalePrice by MasVnrType')
plt.xlabel('MasVnrType')
plt.ylabel('SalePrice')
plt.xticks(rotation=45)
plt.show()

In [ ]:
#MasVnrTypeのダミー変数化
train = pd.get_dummies(train, columns=['MasVnrType'], drop_first=True)
test = pd.get_dummies(test, columns=['MasVnrType'], drop_first=True)

In [ ]:
#現在の学習データを確認
train.head()

In [ ]:
#現在の相関を確認する
plt.figure(figsize=(20,20))
sns.heatmap(train.corr(numeric_only=True), annot=True)

In [ ]:
#現在の定性データとSalePriceの相関を確認
def correlation_ratio(categories, values):
    """カテゴリ（定性）と数値（定量）の関係を測る相関比ηを返す"""
    fcat, _ = pd.factorize(categories)
    mean_per_cat = np.array([values[fcat == i].mean() for i in range(len(np.unique(fcat)))])
    n_per_cat = np.array([np.sum(fcat == i) for i in range(len(np.unique(fcat)))])
    overall_mean = values.mean()
    numerator = np.sum(n_per_cat * (mean_per_cat - overall_mean)**2)
    denominator = np.sum((values - overall_mean)**2)
    return np.sqrt(numerator / denominator) if denominator != 0 else 0

# 定性データの列を抽出
cat_cols = train.select_dtypes(include='object').columns

# 目的変数（定量データ）
target = 'SalePrice'

# 各カテゴリ列との相関比を計算
results = []
for col in cat_cols:
    eta = correlation_ratio(train[col], train[target])
    results.append((col, eta))

# 結果をDataFrame化
corr_eta_df = pd.DataFrame(results, columns=['Column', 'Eta']).sort_values(by='Eta', ascending=False)
corr_eta_df

In [ ]:
#次にFoundationを確認
#まずはデータの中身を確認

train.groupby('Foundation').size()

In [ ]:
#BrkTil, Slab, Stone, Woodについては全体に占める数が少ないため、Otherでまとめる
mapping = {
    'BrkTil':'Other',
    'CBlock':'CBlock',
    'PConc':'PConc',
    'Slab':'Other',
    'Stone':'Other',
    'Wood':'Other'
}

train['Foundation'] = train['Foundation'].replace(mapping)

In [ ]:
#テストデータも同様に実施
mapping = {
    'BrkTil':'Other',
    'CBlock':'CBlock',
    'PConc':'PConc',
    'Slab':'Other',
    'Stone':'Other',
    'Wood':'Other'
}

test['Foundation'] = test['Foundation'].replace(mapping)

In [ ]:
#そのうえでSalePriceとの相関を見てみる
#箱ひげ図で可視化する

plt.figure(figsize=(12,6))
sns.boxplot(data=train, x='Foundation', y='SalePrice')
plt.title('Distribution of SalePrice by Foundation')
plt.xlabel('Foundation')
plt.ylabel('SalePrice')
plt.xticks(rotation=45)
plt.show()

In [ ]:
#FoundationについてはOther→CBlock→PConcの順でsalePriceが高くなるため、ここも機械学習に有効ではないかと考える
#ダミー変数化して、機械学習に用いる

train = pd.get_dummies(train, columns=['Foundation'], drop_first=True)
test = pd.get_dummies(test, columns=['Foundation'], drop_first=True)

In [ ]:
#現在の学習データを確認
train.head()

In [ ]:
#GarageFinishを確認
#まずはデータの分布を確認

train.groupby('GarageFinish').size()

In [ ]:
#garageFinish別のSalePriceを確認

train.groupby('GarageFinish')['SalePrice'].describe()

In [ ]:
#箱ひげ図で可視化する
#以下の傾向が見せるように完成度が高くなるほど、salePriceも高くなる
#こちらもダミー変数化する

plt.figure(figsize=(12,6))
sns.boxplot(data=train, x='GarageFinish', y='SalePrice')
plt.title('Distribution of SalePrice by GarageFinish')
plt.xlabel('GarageFinish')
plt.ylabel('SalePrice')
plt.xticks(rotation=45)
plt.show()

In [ ]:
#ダミー変数化
train = pd.get_dummies(train, columns=['GarageFinish'], drop_first=True)
test = pd.get_dummies(test, columns=['GarageFinish'], drop_first=True)

In [ ]:
#現在の学習データを確認
train.head()

In [ ]:
#定性データの最後にNeighborhoodを確認
#まずNeighborhoodと相関が高い項目をしらべる
#NeighborhoodとSalePriceの相関が高いことはわかっているが、地域以外にも官営する要素があるかは確認
#以下を見ているとYearBlt（建設年）と相関があり、建設年の新しさが地域形成された時期にもつながり、SalePriceの高低につながっている可能性もある

def correlation_ratio(categories, values):
    mask = ~pd.isna(categories) & ~pd.isna(values)
    categories = categories[mask]
    values = values[mask]

    cat_means = values.groupby(categories).mean()
    overall_mean = values.mean()
    n = values.groupby(categories).count()

    numerator = sum(n * (cat_means - overall_mean)**2)
    denominator = sum((values - overall_mean)**2)

    return np.sqrt(numerator / denominator)

cat_col = 'Neighborhood'
num_cols = train.select_dtypes(include=['int64','float64']).columns

eta_results = {}

for col in num_cols:
    eta = correlation_ratio(train[cat_col], train[col])
    eta_results[col] = eta

eta_df = pd.DataFrame.from_dict(eta_results, orient='index', columns=['eta']).sort_values('eta', ascending=False)
eta_df.head(20)

In [ ]:
#めNeighborhoodをワンホットエンコーディング化する
#カテゴリー数が多いため、Frecency Encodingを用いる

freq = train['Neighborhood'].value_counts() / len(train)
train['Neighborhood'] = train['Neighborhood'].map(freq)

In [ ]:
#テストデータも同様に
freq = test['Neighborhood'].value_counts() / len(test)
test['Neighborhood'] = test['Neighborhood'].map(freq)

In [ ]:
#現在の学習データを確認
train.head()

In [ ]:
#定性データの処理は完了しているため、定量データの相関のみを確認
#現在の相関を確認する
plt.figure(figsize=(20,20))
sns.heatmap(train.corr(numeric_only=True), annot=True)

In [ ]:
#家の補助要素的存在の評価（ExterQual, KicthenQual, FireplaceQu, HeatingQC)については、合算して1つの項目とする

train['Supplement_score'] = train['ExterQual_score'] + train['KitchenQual_score'] + train['FireplaceQu_score'] + train['HeatingQC_score']
test['Supplement_score'] = test['ExterQual_score'] + test['KitchenQual_score'] + test['FireplaceQu_score'] + test['HeatingQC_score']

In [ ]:
#Supplement_scoreで用いた元のデータ項目は削除

del train['ExterQual_score']
del train['KitchenQual_score']
del train['FireplaceQu_score']
del train['HeatingQC_score']

del test['ExterQual_score']
del test['KitchenQual_score']
del test['FireplaceQu_score']
del test['HeatingQC_score']

In [ ]:
#学習データおよびテストデータの文字列データを数値化
cat_cols = train.select_dtypes(include='object').columns

train[cat_cols] = train[cat_cols].astype('category')
train[cat_cols] = train[cat_cols].apply(lambda x: x.cat.codes)

test[cat_cols] = test[cat_cols].astype('category')
test[cat_cols] = test[cat_cols].apply(lambda x: x.cat.codes)

In [ ]:
#現在の相関を確認する
plt.figure(figsize=(20,20))
sns.heatmap(train.corr(numeric_only=True), annot=True)

In [ ]:
#相関の低いBsmtQualは削除
del train['BsmtQual']
del test['BsmtQual']

In [ ]:
#重要度が0.001未満の項目を落とす
cols_to_drop = ['Foundation_Other', 
                 'MasVnrType_Detched', 'Foundation_PConc',
                'GarageFinish_RFn', 'GarageFinish_Unf', 'GarageType_Finish',
               'GarageType_Other','GarageFinish_None','MasVnrType_None']

train = train.drop(columns=[c for c in cols_to_drop if c in train.columns])

In [ ]:
#XGboostで確認
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV, train_test_split

# 目的変数と説明変数
X = train.drop('SalePrice', axis=1)
y = train['SalePrice']

# 訓練データと検証データに分割
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=0
)

# モデル
rf = XGBRegressor(random_state=0, n_jobs=-1)

# 探索するパラメータの範囲
param_grid = {
    'n_estimators': [100, 200, 300, 500, 1000],
    'max_depth': [4, 6, 8,10],
    'learning_rate': [0.01, 0.05, 0.1,0.2],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0]
}

model = XGBRegressor(objective='reg:squarederror', random_state=0)

grid = GridSearchCV(model, param_grid, cv=3, scoring='neg_root_mean_squared_error')
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
best_model = grid.best_estimator_


In [ ]:
#モデルの作成
model = XGBRegressor(colsample_bytree = 0.7, learning_rate = 0.01, max_depth= 4, n_estimators= 500, subsample= 0.7, objective = 'reg:squarederror', random_state=0)
model.fit(X_train, y_train)

In [ ]:
# 予測
pred_valid = model.predict(X_valid)

#正確性の算出
from sklearn.metrics import r2_score

r2 = r2_score(y_valid, pred_valid)
print(f"R² Score: {r2:.4f}")

In [ ]:
#テストデータのIDをインデックス化
test = test.set_index('Id')

In [ ]:
#現在のテストデータにおける欠損値を確認
test.isnull().sum()

In [ ]:
#GarageArea, GarageAgeでNAとなっている場所を確認
NA = test[test['GarageArea'].isna()]
NA

In [ ]:
#GarageAgeが0であることから、Garageが存在しないため欠損値となっていると考える
#この欠損値については0と表示して埋める

test.loc[test['GarageArea'].isna(), 'GarageArea'] = 0
test.loc[test['GarageCars'].isna(), 'GarageCars'] = 0

In [ ]:
#テストで使う項目の指定
test_features = test[['House_SF', 'OverallQual', 'YearBuilt', 'MasVnrArea','YearRemodAdd', 'Supplement_score', 'f_BsmtSF','entrance', 'GarageArea',
                     'GarageCars','TotalRoom','Neighborhood','FullBath','MasVnrType_Stone','GarageAge','GarageType_Detchd']].values

In [ ]:
#テストで予測する目的変数
predict_test_target = model.predict(test_features)

In [ ]:
submission = pd.DataFrame({'Id': test.index, 'SalePrice': predict_test_target})
submission.to_csv('submission_Housing.csv', index = False )